# Model Optimization: Hyperparameter Tuning

This notebook performs hyperparameter optimization using:
1. **GridSearchCV** - Exhaustive parameter search
2. **RandomizedSearchCV** - Stochastic parameter sampling
3. **Bayesian Optimization (Optuna)** - Sequential model-based optimization

**Models**: Logistic Regression, Random Forest, XGBoost, SVM

## Section 1: Setup and Data Preparation

In [ ]:
import pandas as pd
import numpy as np
import re, warnings, time, os, json
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (train_test_split, GridSearchCV, RandomizedSearchCV,
    StratifiedKFold, cross_val_score)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix, roc_curve, precision_recall_curve)
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from scipy.sparse import hstack, csr_matrix
import joblib
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
print('Libraries loaded.')
from scipy.stats import uniform, randint, loguniform
try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    HAS_OPTUNA=True
except ImportError:
    HAS_OPTUNA=False
    print('optuna not installed. pip install optuna')

In [ ]:
df = pd.read_csv('../data/cumulative_ai_customer_communication_dataset.csv', low_memory=False)
df['issue_reported_at'] = pd.to_datetime(df['issue_reported_at'], errors='coerce', dayfirst=True)
df['issue_responded'] = pd.to_datetime(df['issue_responded'], errors='coerce', dayfirst=True)
df['target'] = (df['csat_score'] >= 4).astype(int)
print(f'Dataset: {df.shape[0]} rows, Target positive rate: {df["target"].mean()*100:.1f}%')

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
def preprocess_text(text):
    if pd.isna(text) or not isinstance(text, str): return ''
    text = text.lower()
    text = text.encode('ascii','ignore').decode('ascii')
    text = re.sub(r'[^a-z\s]','',text)
    text = re.sub(r'\s+',' ',text).strip()
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words and len(t)>1]
    return ' '.join(tokens)
print('Preprocessing text...')
df['cleaned_message'] = df['customer_message'].apply(preprocess_text)
print(f'Done. Non-empty: {(df["cleaned_message"]!="").sum()}')

In [ ]:
df['response_time_minutes'] = ((df['issue_responded']-df['issue_reported_at']).dt.total_seconds()/60).clip(lower=0).fillna(0)
df['issue_hour'] = df['issue_reported_at'].dt.hour.fillna(0).astype(int)
df['issue_day_of_week'] = df['issue_reported_at'].dt.dayofweek.fillna(0).astype(int)
for col,src in [('channel_encoded','channel_name'),('category_encoded','category'),
                ('subcategory_encoded','sub-category'),('shift_encoded','agent_shift')]:
    le=LabelEncoder(); df[col]=le.fit_transform(df[src].fillna('Unknown'))
tenure_map={'On Job Training':0,'0-30':1,'31-60':2,'61-90':3,'>90':4}
df['tenure_encoded']=df['tenure_bucket'].map(tenure_map).fillna(0).astype(int)
df['has_message']=(df['cleaned_message']!='').astype(int)
df['cleaned_word_count']=df['cleaned_message'].apply(lambda x:len(x.split()) if x else 0)
structured_features=['response_time_minutes','issue_hour','issue_day_of_week','channel_encoded',
    'category_encoded','subcategory_encoded','shift_encoded','tenure_encoded',
    'message_length','word_count','has_message','cleaned_word_count']
print(f'Structured features: {len(structured_features)}')

In [ ]:
X_structured = df[structured_features].fillna(0)
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X_structured, y, test_size=0.2, random_state=42, stratify=y)
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2), min_df=5)
X_text_train = tfidf.fit_transform(df.loc[X_train.index,'cleaned_message'])
X_text_test = tfidf.transform(df.loc[X_test.index,'cleaned_message'])
X_combined_train = hstack([X_text_train, csr_matrix(X_train.values)])
X_combined_test = hstack([X_text_test, csr_matrix(X_test.values)])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
neg_count=(y_train==0).sum(); pos_count=(y_train==1).sum(); scale_weight=neg_count/pos_count
print(f'Train:{X_train.shape[0]}, Test:{X_test.shape[0]}, TF-IDF:{X_text_train.shape[1]}, Combined:{X_combined_train.shape[1]}')

## Section 2: GridSearchCV

In [ ]:
# GridSearchCV: Logistic Regression
print('=== GridSearchCV: Logistic Regression ===')
lr_param_grid = {'C':[0.01,0.1,1,10,100],'penalty':['l1','l2'],'solver':['liblinear','saga'],'class_weight':['balanced',None]}
start=time.time()
lr_grid=GridSearchCV(LogisticRegression(max_iter=1000,random_state=42),lr_param_grid,cv=cv,scoring='f1',n_jobs=-1)
lr_grid.fit(X_combined_train, y_train)
print(f'Best params: {lr_grid.best_params_}')
print(f'Best CV F1: {lr_grid.best_score_*100:.2f}%  Time: {time.time()-start:.1f}s')
y_pred=lr_grid.predict(X_combined_test)
print(f'Test Acc: {accuracy_score(y_test,y_pred)*100:.2f}%, F1: {f1_score(y_test,y_pred)*100:.2f}%')

In [ ]:
# GridSearchCV: Random Forest
print('=== GridSearchCV: Random Forest ===')
rf_param_grid = {'n_estimators':[100,200,300],'max_depth':[10,20,30,None],'min_samples_split':[2,5,10],'min_samples_leaf':[1,2,4],'class_weight':['balanced','balanced_subsample']}
start=time.time()
rf_grid=GridSearchCV(RandomForestClassifier(random_state=42,n_jobs=-1),rf_param_grid,cv=cv,scoring='f1',n_jobs=-1)
rf_grid.fit(X_text_train, y_train)
print(f'Best params: {rf_grid.best_params_}')
print(f'Best CV F1: {rf_grid.best_score_*100:.2f}%  Time: {time.time()-start:.1f}s')
y_pred=rf_grid.predict(X_text_test)
print(f'Test Acc: {accuracy_score(y_test,y_pred)*100:.2f}%, F1: {f1_score(y_test,y_pred)*100:.2f}%')

In [ ]:
# GridSearchCV: XGBoost
print('=== GridSearchCV: XGBoost ===')
xgb_param_grid = {'n_estimators':[100,200,300],'max_depth':[3,5,7,9],'learning_rate':[0.01,0.05,0.1,0.2],'subsample':[0.8,1.0],'colsample_bytree':[0.8,1.0]}
start=time.time()
xgb_grid=GridSearchCV(XGBClassifier(scale_pos_weight=scale_weight,random_state=42,eval_metric='logloss',use_label_encoder=False),xgb_param_grid,cv=cv,scoring='f1',n_jobs=-1)
xgb_grid.fit(X_train, y_train)
print(f'Best params: {xgb_grid.best_params_}')
print(f'Best CV F1: {xgb_grid.best_score_*100:.2f}%  Time: {time.time()-start:.1f}s')
y_pred=xgb_grid.predict(X_test)
print(f'Test Acc: {accuracy_score(y_test,y_pred)*100:.2f}%, F1: {f1_score(y_test,y_pred)*100:.2f}%')

## Section 3: RandomizedSearchCV

In [ ]:
# RandomizedSearchCV: XGBoost (larger space)
print('=== RandomizedSearchCV: XGBoost ===')
xgb_dist = {'n_estimators':randint(50,500),'max_depth':randint(3,12),'learning_rate':loguniform(0.005,0.5),
    'subsample':uniform(0.6,0.4),'colsample_bytree':uniform(0.6,0.4),'min_child_weight':randint(1,10),
    'gamma':uniform(0,0.5),'reg_alpha':loguniform(1e-4,10),'reg_lambda':loguniform(1e-4,10)}
start=time.time()
xgb_random=RandomizedSearchCV(XGBClassifier(scale_pos_weight=scale_weight,random_state=42,eval_metric='logloss',use_label_encoder=False),
    xgb_dist,n_iter=100,cv=cv,scoring='f1',n_jobs=-1,random_state=42)
xgb_random.fit(X_train, y_train)
print(f'Best params: {xgb_random.best_params_}')
print(f'Best CV F1: {xgb_random.best_score_*100:.2f}%  Time: {time.time()-start:.1f}s (100 iters)')
y_pred=xgb_random.predict(X_test)
print(f'Test Acc: {accuracy_score(y_test,y_pred)*100:.2f}%, F1: {f1_score(y_test,y_pred)*100:.2f}%')

## Section 4: Bayesian Optimization (Optuna)

In [ ]:
if HAS_OPTUNA:
    print('=== Bayesian Optimization: XGBoost ===')
    def objective(trial):
        p={'n_estimators':trial.suggest_int('n_estimators',50,500),
           'max_depth':trial.suggest_int('max_depth',3,12),
           'learning_rate':trial.suggest_float('learning_rate',0.005,0.5,log=True),
           'subsample':trial.suggest_float('subsample',0.6,1.0),
           'colsample_bytree':trial.suggest_float('colsample_bytree',0.6,1.0),
           'min_child_weight':trial.suggest_int('min_child_weight',1,10),
           'gamma':trial.suggest_float('gamma',0,0.5),
           'reg_alpha':trial.suggest_float('reg_alpha',1e-4,10,log=True),
           'reg_lambda':trial.suggest_float('reg_lambda',1e-4,10,log=True)}
        model=XGBClassifier(**p,scale_pos_weight=scale_weight,random_state=42,eval_metric='logloss',use_label_encoder=False)
        return cross_val_score(model,X_train,y_train,cv=cv,scoring='f1',n_jobs=-1).mean()
    start=time.time()
    study=optuna.create_study(direction='maximize',sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(objective,n_trials=50,show_progress_bar=False)
    print(f'Best F1: {study.best_value*100:.2f}%, Time: {time.time()-start:.1f}s')
    print(f'Best params: {study.best_params}')
    best_xgb=XGBClassifier(**study.best_params,scale_pos_weight=scale_weight,random_state=42,eval_metric='logloss',use_label_encoder=False)
    best_xgb.fit(X_train,y_train)
    y_pred=best_xgb.predict(X_test)
    print(f'Test Acc: {accuracy_score(y_test,y_pred)*100:.2f}%, F1: {f1_score(y_test,y_pred)*100:.2f}%')
else:
    print('Optuna not available - skipping Bayesian optimization')

## Section 5: Save Optimized Models & Summary

In [ ]:
# Save optimized models
os.makedirs('../models', exist_ok=True)
joblib.dump(lr_grid.best_estimator_, '../models/optimized_lr.pkl')
joblib.dump(rf_grid.best_estimator_, '../models/optimized_rf.pkl')
joblib.dump(xgb_grid.best_estimator_, '../models/optimized_xgb_grid.pkl')
if HAS_OPTUNA: joblib.dump(best_xgb, '../models/optimized_xgb_bayesian.pkl')
print('Models saved to ../models/')

# Summary visualization
results_data = {'Method':['Grid-LR','Grid-RF','Grid-XGB','Random-XGB'],
    'F1':[lr_grid.best_score_,rf_grid.best_score_,xgb_grid.best_score_,xgb_random.best_score_]}
if HAS_OPTUNA: results_data['Method'].append('Bayes-XGB'); results_data['F1'].append(study.best_value)
rdf=pd.DataFrame(results_data)
fig,axes=plt.subplots(1,2,figsize=(14,5))
sns.barplot(data=rdf,x='Method',y='F1',palette='viridis',ax=axes[0])
axes[0].set_title('Optimization Results: Best CV F1');axes[0].set_ylabel('F1 Score')
axes[0].set_ylim(0,1)
if HAS_OPTUNA:
    vals=[t.value for t in study.trials if t.value]; best_so_far=[max(vals[:i+1]) for i in range(len(vals))]
    axes[1].plot(vals,'o',alpha=0.4,label='Trial');axes[1].plot(best_so_far,'-r',lw=2,label='Best')
    axes[1].set_title('Bayesian Optimization Convergence');axes[1].set_xlabel('Trial');axes[1].set_ylabel('F1');axes[1].legend()
else: axes[1].text(0.5,0.5,'Optuna unavailable',ha='center',va='center')
plt.tight_layout();plt.savefig('../models/optimization_results.png',dpi=150,bbox_inches='tight');plt.show()
print('Done.')